In [9]:
import numpy as np
import matplotlib.pyplot as plt
import orekit
from orekit.pyhelpers import setup_orekit_curdir
from org.orekit.utils import Constants

# ==========================================
# 1. INIZIALIZZAZIONE AMBIENTE OREKIT
# ==========================================
orekit.initVM()
setup_orekit_curdir('C:/Users/Simone/Downloads/orekit-data-main')

# ==========================================
# 2. COSTANTI E DATI DI INPUT
# ==========================================
# Costanti Fisiche Terra (da Orekit, convertite in km)
mu_E = Constants.WGS84_EARTH_MU / 1e9             # Parametro gravitazionale Terrestre [km^3/s^2]
R_E = Constants.WGS84_EARTH_EQUATORIAL_RADIUS / 1000.0  # Raggio medio terrestre [km]

# Costanti Fisiche Luna (Esplicite per Patched Conics)
mu_M = 4902.8000                # Parametro gravitazionale Lunare [km^3/s^2]
R_M = 1737.4                    # Raggio medio lunare [km]
D_EM = 384400.0                 # Distanza media Terra-Luna [km]

# Parametri Orbita Iniziale (LEO - Low Earth Orbit)
h_LEO = 300.0                   # Quota di parcheggio terrestre [km]
r1 = R_E + h_LEO                # Raggio orbita iniziale [km]
v_LEO = np.sqrt(mu_E / r1)      # Velocità circolare LEO [km/s]

# Parametri Orbita Finale (LLO - Low Lunar Orbit)
h_LLO = 100.0                   # Quota di parcheggio lunare [km]
r2 = R_M + h_LLO                # Raggio orbita lunare target [km]
v_LLO = np.sqrt(mu_M / r2)      # Velocità circolare LLO [km/s]

print("========================================================")
print("  1. PARAMETRI DELLE ORBITE DI PARCHEGGIO")
print("========================================================")
print(f"Velocita in LEO (Terra): {v_LEO:.4f} km/s")
print(f"Velocita in LLO (Luna):  {v_LLO:.4f} km/s\n")

# ==========================================
# 3. FASE 1: TRANS-LUNAR INJECTION (TLI)
# ==========================================
r_apogeo_tx = D_EM
a_tx = (r1 + r_apogeo_tx) / 2.0  # Semiasse maggiore del trasferimento

# Calcolo velocità sull'ellisse (Equazione della vis-viva)
v_tx_perigeo = np.sqrt(mu_E * (2/r1 - 1/a_tx))
v_tx_apogeo = np.sqrt(mu_E * (2/r_apogeo_tx - 1/a_tx))

# Delta V della manovra di partenza
DeltaV_TLI = v_tx_perigeo - v_LEO

# Tempo di volo (Solo andata, quindi metà periodo orbitale)
TOF_sec = np.pi * np.sqrt((a_tx**3) / mu_E)
TOF_days = TOF_sec / 86400.0

print("========================================================")
print("  2. PARTENZA DALLA TERRA (TRANS-LUNAR INJECTION)")
print("========================================================")
print(f"Velocita necessaria al perigeo: {v_tx_perigeo:.4f} km/s")
print(f"Delta V TLI (Burn 1):           {DeltaV_TLI:.4f} km/s")
print(f"Tempo di Volo (TOF):            {TOF_days:.2f} giorni\n")

# ==========================================
# 4. FASE 2: ARRIVO E LUNAR ORBIT INSERTION (LOI)
# ==========================================
v_Moon = np.sqrt(mu_E / D_EM)     # Velocità orbitale della Luna [km/s]
v_inf = v_Moon - v_tx_apogeo      # Eccesso iperbolico (V_infinity) [km/s]

# Conservazione dell'energia per l'iperbole lunare
v_iperbole_perilenio = np.sqrt(v_inf**2 + 2*mu_M / r2)

# Delta V per la cattura
DeltaV_LOI = v_iperbole_perilenio - v_LLO

print("========================================================")
print("  3. ARRIVO ALLA LUNA (LUNAR ORBIT INSERTION)")
print("========================================================")
print(f"Velocita della Luna:              {v_Moon:.4f} km/s")
print(f"Eccesso Iperbolico (V_inf):       {v_inf:.4f} km/s")
print(f"Vel. Iperbole al perilenio (r2):  {v_iperbole_perilenio:.4f} km/s")
print(f"Delta V LOI (Burn 2 - Frenata):   {DeltaV_LOI:.4f} km/s\n")

# ==========================================
# 5. RISULTATI FINALI BUDGET DELTA-V
# ==========================================
DeltaV_Tot = DeltaV_TLI + DeltaV_LOI

print("========================================================")
print("  BUDGET DELTA-V TOTALE ")
print("========================================================")
print(f"Delta V Partenza:  {DeltaV_TLI:.4f} km/s")
print(f"Delta V Cattura:   {DeltaV_LOI:.4f} km/s")
print(f"DELTA V TOTALE:    {DeltaV_Tot:.4f} km/s")
print("========================================================")


  1. PARAMETRI DELLE ORBITE DI PARCHEGGIO
Velocita in LEO (Terra): 7.7258 km/s
Velocita in LLO (Luna):  1.6335 km/s

  2. PARTENZA DALLA TERRA (TRANS-LUNAR INJECTION)
Velocita necessaria al perigeo: 10.8322 km/s
Delta V TLI (Burn 1):           3.1064 km/s
Tempo di Volo (TOF):            4.98 giorni

  3. ARRIVO ALLA LUNA (LUNAR ORBIT INSERTION)
Velocita della Luna:              1.0183 km/s
Eccesso Iperbolico (V_inf):       0.8301 km/s
Vel. Iperbole al perilenio (r2):  2.4547 km/s
Delta V LOI (Burn 2 - Frenata):   0.8212 km/s

  BUDGET DELTA-V TOTALE 
Delta V Partenza:  3.1064 km/s
Delta V Cattura:   0.8212 km/s
DELTA V TOTALE:    3.9277 km/s
